### Rag pipelines- Data ingestion to Vector DB pipeline


In [28]:
import os

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [29]:
### read all the pdfs inside the directory 
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []

    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: pdf1.pdf
✓ Loaded 48 pages

Processing: pdf2.pdf
✓ Loaded 94 pages

Processing: pdf3.pdf
✓ Loaded 36 pages

Total documents loaded: 178


In [30]:
all_pdf_documents


[Document(metadata={'producer': 'Samsung Electronics', 'creator': 'Samsung Electronics', 'creationdate': 'D:', 'moddate': '2024-03-14T16:00:00+05:30', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='NPTEL Online Certification Courses \nIndian Institute of Technology Kharagpur \n \n \nIntroduction to \nInternet of Things \nAssignment-Week 1 \nTYPE OF QUESTION: MCQ/MSQ \nNumber of questions: 15 Total marks: 15 X 1= 15 \n \n \n \n \nQUESTION 1: \nWhat is the full form of IoT? \na. Internet of Tasks \nb. Internet of Things \nc. Internet of Tasks \nd. None of these \nCorrect Answer: b. Internet of Things \nDetailed Solution: The full form of IoT is “Internet of Things” \n \nSee lecture 1 (Introduction to IoT – Part - I) @ 1:30 \n \n \nQUESTION 2: \nWhich of the following technologies have unified and has resulted in the evolution of IoT? \na. Low-power embedded systems \nb. Cloud Computing \n

In [31]:
### text splitting into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


# Create chunks
chunks = split_documents(all_pdf_documents)

# Display chunks
chunks

Split 178 documents into 227 chunks

Example chunk:
Content: NPTEL Online Certification Courses 
Indian Institute of Technology Kharagpur 
 
 
Introduction to 
Internet of Things 
Assignment-Week 1 
TYPE OF QUESTION: MCQ/MSQ 
Number of questions: 15 Total marks...
Metadata: {'producer': 'Samsung Electronics', 'creator': 'Samsung Electronics', 'creationdate': 'D:', 'moddate': '2024-03-14T16:00:00+05:30', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Samsung Electronics', 'creator': 'Samsung Electronics', 'creationdate': 'D:', 'moddate': '2024-03-14T16:00:00+05:30', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='NPTEL Online Certification Courses \nIndian Institute of Technology Kharagpur \n \n \nIntroduction to \nInternet of Things \nAssignment-Week 1 \nTYPE OF QUESTION: MCQ/MSQ \nNumber of questions: 15 Total marks: 15 X 1= 15 \n \n \n \n \nQUESTION 1: \nWhat is the full form of IoT? \na. Internet of Tasks \nb. Internet of Things \nc. Internet of Tasks \nd. None of these \nCorrect Answer: b. Internet of Things \nDetailed Solution: The full form of IoT is “Internet of Things” \n \nSee lecture 1 (Introduction to IoT – Part - I) @ 1:30 \n \n \nQUESTION 2: \nWhich of the following technologies have unified and has resulted in the evolution of IoT? \na. Low-power embedded systems \nb. Cloud Computing \n

In [32]:
chunks = split_documents(all_pdf_documents)

print(len(chunks))

Split 178 documents into 227 chunks

Example chunk:
Content: NPTEL Online Certification Courses 
Indian Institute of Technology Kharagpur 
 
 
Introduction to 
Internet of Things 
Assignment-Week 1 
TYPE OF QUESTION: MCQ/MSQ 
Number of questions: 15 Total marks...
Metadata: {'producer': 'Samsung Electronics', 'creator': 'Samsung Electronics', 'creationdate': 'D:', 'moddate': '2024-03-14T16:00:00+05:30', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}
227


#### Embedding And VectorstoreDB


In [33]:
import numpy as np

from sentence_transformers import SentenceTransformer

import chromadb
from chromadb.config import Settings

import uuid

from typing import List, Dict, Any, Tuple

from sklearn.metrics.pairwise import cosine_similarity

In [34]:
import numpy as np
from typing import List
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(
                f"Model loaded successfully. Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings

## initialize the embedding manager

embedding_manager = EmbeddingManager()

embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12263.00it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21528\3553525902.py:30: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


### vectorstore

In [35]:
import os
import uuid
import numpy as np
import chromadb

from typing import List, Any

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: {self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents to vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(
                doc.page_content
            )

            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(
                embedding.tolist()
            )

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} documents to vector store"
            )

            print(
                f"Total documents in collection: {self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise

VectorStore=VectorStore()
VectorStore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 681


In [36]:
chunks

[Document(metadata={'producer': 'Samsung Electronics', 'creator': 'Samsung Electronics', 'creationdate': 'D:', 'moddate': '2024-03-14T16:00:00+05:30', 'source': '..\\data\\pdf\\pdf1.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='NPTEL Online Certification Courses \nIndian Institute of Technology Kharagpur \n \n \nIntroduction to \nInternet of Things \nAssignment-Week 1 \nTYPE OF QUESTION: MCQ/MSQ \nNumber of questions: 15 Total marks: 15 X 1= 15 \n \n \n \n \nQUESTION 1: \nWhat is the full form of IoT? \na. Internet of Tasks \nb. Internet of Things \nc. Internet of Tasks \nd. None of these \nCorrect Answer: b. Internet of Things \nDetailed Solution: The full form of IoT is “Internet of Things” \n \nSee lecture 1 (Introduction to IoT – Part - I) @ 1:30 \n \n \nQUESTION 2: \nWhich of the following technologies have unified and has resulted in the evolution of IoT? \na. Low-power embedded systems \nb. Cloud Computing \n

In [37]:
split_documents

<function __main__.split_documents(documents, chunk_size=1000, chunk_overlap=200)>

In [38]:
### convert the text to embedding
texts=[doc.page_content for doc in chunks]
texts

['NPTEL Online Certification Courses \nIndian Institute of Technology Kharagpur \n \n \nIntroduction to \nInternet of Things \nAssignment-Week 1 \nTYPE OF QUESTION: MCQ/MSQ \nNumber of questions: 15 Total marks: 15 X 1= 15 \n \n \n \n \nQUESTION 1: \nWhat is the full form of IoT? \na. Internet of Tasks \nb. Internet of Things \nc. Internet of Tasks \nd. None of these \nCorrect Answer: b. Internet of Things \nDetailed Solution: The full form of IoT is “Internet of Things” \n \nSee lecture 1 (Introduction to IoT – Part - I) @ 1:30 \n \n \nQUESTION 2: \nWhich of the following technologies have unified and has resulted in the evolution of IoT? \na. Low-power embedded systems \nb. Cloud Computing \nc. Machine Learning \nd. All of these \nCorrect Answer: d. All of these \nDetailed Solution: Unification of technologies which has resulted in the advancement of \nIoT are –  \na. Low-power embedded systems \nb. Cloud Computing \nc. Big Data \nd. Machine Learning \ne. Networking \nSee lecture 1 (

In [39]:
### convert the text to embedding
texts=[doc.page_content for doc in chunks]

### Generate the Embedding

embeddings=embedding_manager.generate_embeddings(texts)

### store in the vector database

VectorStore.add_documents(chunks,embeddings)

Generating embeddings for 227 texts...


Batches: 100%|██████████| 8/8 [00:07<00:00,  1.09it/s]


Generated embeddings with shape: (227, 384)
Adding 227 documents to vector store...
Successfully added 227 documents to vector store
Total documents in collection: 908


### Rag Ratriver pipeline from VectorStore

In [40]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # Convert distance to similarity score
                    # ChromaDB uses cosine distance
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:

                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1
                            }
                        )

                print(
                    f"Retrieved {len(retrieved_docs)} documents "
                    f"(after filtering)"
                )

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever=RAGRetriever(VectorStore,embedding_manager)
rag_retriever

In [41]:
rag_retriever.retrieve("nptel")

Retrieving documents for query: 'nptel'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 88.02it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e3352370_26',
  'content': 'NPTEL Online Certification Courses \nIndian Institute of Technology Kharagpur \n \n \n************END***********',
  'metadata': {'page': 22,
   'moddate': '2024-03-14T16:00:00+05:30',
   'creationdate': 'D:',
   'total_pages': 48,
   'content_length': 108,
   'source': '..\\data\\pdf\\pdf1.pdf',
   'file_type': 'pdf',
   'source_file': 'pdf1.pdf',
   'doc_index': 26,
   'producer': 'Samsung Electronics',
   'creator': 'Samsung Electronics',
   'page_label': '23'},
  'similarity_score': 0.3256518840789795,
  'distance': 0.6743481159210205,
  'rank': 1},
 {'id': 'doc_1d275f03_26',
  'content': 'NPTEL Online Certification Courses \nIndian Institute of Technology Kharagpur \n \n \n************END***********',
  'metadata': {'page': 22,
   'source': '..\\data\\pdf\\pdf1.pdf',
   'page_label': '23',
   'source_file': 'pdf1.pdf',
   'creationdate': 'D:',
   'content_length': 108,
   'moddate': '2024-03-14T16:00:00+05:30',
   'total_pages': 48,
   'doc

In [42]:
rag_retriever.retrieve("IOT")

Retrieving documents for query: 'IOT'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 67.48it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d53662d1_9',
  'content': 'NPTEL Online Certification Courses \nIndian Institute of Technology Kharagpur \n \n \nDetailed Solution: A sensor node is made up of a combination of sensor/sensors, a processor \nunit, a radio unit, and a power unit. \n \nSee Page number – 101, Chapter - 5, Book - Introduction to IoT, Authors – Sudip Misra, \nAnandarup Mukherjee, and Arijit Roy, Publisher – Cambridge University Press, Edition – 1 \n(2021) \n \n \n \n************END***********',
  'metadata': {'total_pages': 48,
   'creationdate': 'D:',
   'content_length': 434,
   'doc_index': 9,
   'page': 6,
   'file_type': 'pdf',
   'producer': 'Samsung Electronics',
   'source_file': 'pdf1.pdf',
   'creator': 'Samsung Electronics',
   'source': '..\\data\\pdf\\pdf1.pdf',
   'moddate': '2024-03-14T16:00:00+05:30',
   'page_label': '7'},
  'similarity_score': 0.18636780977249146,
  'distance': 0.8136321902275085,
  'rank': 1},
 {'id': 'doc_1f17c61b_9',
  'content': 'NPTEL Online Certification 

### Integration VectorDB Context pipeline With LLM output

In [43]:
### Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os 

In [44]:
from langchain_groq import ChatGroq

print("Groq imported successfully!")

Groq imported successfully!


In [45]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

load_dotenv()

groq_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

print("Groq LLM initialized!")


Groq LLM initialized!


In [46]:
### Simple RAG pipeline with Groq LLM

if groq_llm:

    query = "What is Unified Multi-task Learning Framework?"

    retrieved_docs = rag_retriever.retrieve(
        query,
        top_k=3,
        score_threshold=0.1
    )

    if retrieved_docs:

        # Combine top retrieved documents as context
        combined_context = "\n\n".join(
            [doc['content'] for doc in retrieved_docs]
        )

        # Generate response using Groq LLM
        response = groq_llm.generate_response(
            query,
            combined_context
        )

        print(f"\nResponse:\n{response}")

    else:
        print("No relevant documents found for the query.")

Retrieving documents for query: 'What is Unified Multi-task Learning Framework?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 79.77it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant documents found for the query.


In [47]:
### Simple RAG pipeline with Groq LLM

if groq_llm:

    query = "IOT"

    retrieved_docs = rag_retriever.retrieve(
        query,
        top_k=3,
        score_threshold=0.1
    )

    if retrieved_docs:

        # Combine top retrieved documents as context
        combined_context = "\n\n".join(
            [doc['content'] for doc in retrieved_docs]
        )

        # Create prompt for Groq
        prompt = f"""
        Context:
        {combined_context}

        Question:
        {query}

        Answer based only on the provided context.
        """

        # Generate response using Groq LLM
        response = groq_llm.invoke(prompt)

        print("\nResponse:\n")
        print(response.content)

    else:
        print("No relevant documents found for the query.")

Retrieving documents for query: 'IOT'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 104.10it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)



Response:

A sensor node is made up of a combination of sensor/sensors, a processor unit, a radio unit, and a power unit.


In [48]:
### Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

# Initialize the Groq LLM
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.1,
    max_tokens=1024
)


# 2. Simple RAG function: retrieve context + generate response

def rag_simple(query, retriever, llm, top_k=3):

    results = retriever.retrieve(
        query,
        top_k=top_k
    )

    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

In [49]:
response = llm.invoke("What is the full form of IoT?")

print(response.content)

The full form of IoT is 'Internet of Things'.


### Enhancerd RAG Pipeline Features

In [50]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(query, retriever, llm,
                 top_k=5,
                 min_score=0.2,
                 return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score,
      and optionally full context.
    """

    results = retriever.retrieve(
        query,
        top_k=top_k,
        score_threshold=min_score
    )

    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # Prepare context and sources
    context = "\n\n".join(
        [doc['content'] for doc in results]
    )

    sources = [
        {
            'source': doc['metadata'].get(
                'source_file',
                doc['metadata'].get('source', 'unknown')
            ),
            'page': doc['metadata'].get(
                'page',
                'unknown'
            ),
            'score': doc['similarity_score'],
            'preview': doc['content'][:120] + '...'
        }
        for doc in results
    ]

    confidence = max(
        [doc['similarity_score'] for doc in results]
    )

    # Generate answer
    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context

    return output

### Example usage:

result = rag_advanced(
    "What is  the enablers of IoT?",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is  the enablers of IoT?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 77.81it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The enablers of IoT are:

1. RFID
2. Nanotechnology
3. Sensors
Sources: [{'source': 'pdf2.pdf', 'page': 3, 'score': 0.20972907543182373, 'preview': 'NPTEL Online Certification Courses \nIndian Institute of Technology Kharagpur \n \n \nQUESTION 7: \nWhich of the following is...'}, {'source': 'pdf2.pdf', 'page': 3, 'score': 0.20972907543182373, 'preview': 'NPTEL Online Certification Courses \nIndian Institute of Technology Kharagpur \n \n \nQUESTION 7: \nWhich of the following is...'}, {'source': 'pdf2.pdf', 'page': 3, 'score': 0.20972907543182373, 'preview': 'NPTEL Online Certification Courses \nIndian Institute of Technology Kharagpur \n \n \nQUESTION 7: \nWhich of the following is...'}]
Confidence: 0.20972907543182373
Context Preview: NPTEL Online Certification Courses 
Indian Institute of Technology Kharagpur 
 
 
QUESTION 7: 
Which of the following is/are not enablers of IoT? 
a. Advancement in gene sequencing 
b. Nanotechnology 
c. Sensors 
d. RFID 
Correct Answer: a. Advan

In [51]:
result = rag_advanced(
    "Which of the following are the features of an IoT Gateway? ",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Which of the following are the features of an IoT Gateway? '
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.83it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Based on the given context, the features of an IoT Gateway are:

- Connects the IoT LAN to a WAN (Option a)
- Can implement several LAN and WAN (Option b)

So, the correct answer is: 
c. Both (a) and (b)
Sources: [{'source': 'pdf3.pdf', 'page': 0, 'score': 0.3233886957168579, 'preview': '© Exam Buddy →  NPTEL- JULY 2025 Course Notes                     Gmail – nptelsolver@gmail.com \n \nInternet of Things JU...'}, {'source': 'pdf3.pdf', 'page': 0, 'score': 0.3233886957168579, 'preview': '© Exam Buddy →  NPTEL- JULY 2025 Course Notes                     Gmail – nptelsolver@gmail.com \n \nInternet of Things JU...'}, {'source': 'pdf3.pdf', 'page': 0, 'score': 0.3233886957168579, 'preview': '© Exam Buddy →  NPTEL- JULY 2025 Course Notes                     Gmail – nptelsolver@gmail.com \n \nInternet of Things JU...'}]
Confidence: 0.3233886957168579
Context Preview: © Exam Buddy →  NPTEL- JULY 2025 Course Notes                     Gmail – nptelsolver@gmail.com 
 
Internet of Things 

In [52]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---

from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False
    ) -> Dict[str, Any]:

        # Retrieve relevant documents
        results = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score
        )

        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""

        else:
            context = "\n\n".join(
                [doc['content'] for doc in results]
            )

            sources = [
                {
                    'source': doc['metadata'].get(
                        'source_file',
                        doc['metadata'].get('source', 'unknown')
                    ),
                    'page': doc['metadata'].get(
                        'page',
                        'unknown'
                    ),
                    'score': doc['similarity_score'],
                    'preview': doc['content'][:120] + '...'
                }
                for doc in results
            ]

            # Streaming answer simulation
            prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{question}

Answer:
"""

            if stream:
                print("Streaming answer:")

                for i in range(0, len(prompt), 80):
                    print(prompt[i:i + 80], end="", flush=True)
                    time.sleep(0.05)

                print()

            response = self.llm.invoke(prompt)
            answer = response.content

        # Add citations to answer
        citations = [
            f"[{i+1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]

        answer_with_citations = (
            answer +
            "\n\nCitations:\n" +
            "\n".join(citations)
            if citations else answer
        )

        # Optionally summarize answer
        summary = None

        if summarize and answer:
            summary_prompt = (
                f"Summarize the following answer in 2 sentences:\n\n{answer}"
            )

            summary_resp = self.llm.invoke(summary_prompt)
            summary = summary_resp.content

        # Store query history
        self.history.append(
            {
                'question': question,
                'answer': answer,
                'sources': sources,
                'summary': summary
            }
        )

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

In [53]:
adv_rag = AdvancedRAGPipeline(
    rag_retriever,
    llm
)

result = adv_rag.query(
    "What is positional encoding?",
    top_k=3,
    min_score=0.3,
    stream=True,
    summarize=True
)

print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What is positional encoding?'
Top K: 3, Score threshold: 0.3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 64.66it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)

Final Answer: No relevant context found.
Summary: There is no information to summarize as the provided text does not contain any relevant context.
History: {'question': 'What is positional encoding?', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'There is no information to summarize as the provided text does not contain any relevant context.'}


In [54]:
adv_rag = AdvancedRAGPipeline(
    rag_retriever,
    llm
)

result = adv_rag.query("Which of the following is/are not enablers of IoT?",
    top_k=3,
    min_score=0.3,
    stream=True,
    summarize=True
)

print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Which of the following is/are not enablers of IoT?'
Top K: 3, Score threshold: 0.3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.68it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:

Use the following context to answer the question concisely.

Context:
NPTEL Onl

ine Certification Courses 
Indian Institute of Technology Kharagpur 
 
 
QUESTION 7: 
Which of the following is/are not enablers of IoT? 
a. Advancement in gene sequencing 
b. Nanotechnology 
c. Sensors 
d. RFID 
Correct Answer: a. Advancement in gene sequencing 
Detailed Solution: The enablers of IoT are –  
a. RFID 
b. Nanotechnology 
c. Sensors 
 
See lecture 1 (Introduction to IoT – Part - I) @ 12:41 
 
 
QUESTION 8: 
State whether the following statement is True or False. 
 
Statement: The decreasing number of devices in IoT is expected to result in an address crunch. 
 
a. True 
b. False 
Correct Answer: b. False 
Detailed Solution: The increasing number of devices in IoT is expected to result in an address 
crunch. 
 
See lecture 2 (Introduction to IoT – Part - II) @ 01:19

NPTEL Online Certification Courses 
Indian Institute of Technology Kharagpur 
 
 
QUESTION 7: 
Which of the following is/are not enablers of IoT? 
a. Advancement in gene sequencing 
b. Nanotechnology 
c. Sens